# 01 — Data audit
Reproducible audit of the committed 2019 Tashkent listing dataset. Run from the repository or its `notebooks/` directory.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
df = pd.read_csv(ROOT / 'data' / 'house_prices.csv')
df.head()

## Shape, schema, missingness, and duplicates
These checks establish whether the file is suitable before any model decision.

In [ ]:
print('shape:', df.shape)
display(df.dtypes.rename('dtype').to_frame())
display(df.isna().sum().rename('missing').to_frame())
print('exact duplicates:', df.duplicated().sum())
print('invalid floors:', (df['level'] > df['max_levels']).sum())

**Observation:** Required values and floor relationships are complete, but 696 exact duplicate advertisements must be removed before splitting to prevent leakage.

In [ ]:
display(df.describe(include='all').T)
display(df['district'].value_counts().rename('rows').to_frame())

**Observation:** District coverage is strongly uneven. Bektemir and Yangihayot are too sparse for dependable district-level conclusions, so the split and CV are stratified and slice counts are reported.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df['price'], bins=60, ax=axes[0])
axes[0].set_title('Listing price distribution (USD)')
sns.scatterplot(data=df, x='size', y='price', hue='district', legend=False, alpha=.35, ax=axes[1])
axes[1].set_title('Price vs size')
plt.tight_layout()

**Observation:** Price is right-skewed and contains luxury outliers. They are retained because the source does not prove they are errors; MAE is primary while RMSE exposes their impact. Exact address and any target-derived price-per-m² feature are excluded to limit memorization/leakage.

In [ ]:
from src.data import load_dataset
X, y, audit = load_dataset(ROOT / 'data' / 'house_prices.csv')
audit